# OPTICO-ACS | huCIRA cytokine response enrichment, Rupture vs Erosion

**Panels:** Fig. 4b (heatmap), Fig. 4c-e (NES per cytokine, plotted in Prism)  
**Input:** `Optico_revisions/data/optico_final.h5ad` (h5ad export of `data/optico_final.rds`)  
**Output:** `Optico_revisions/tables/HuCIRA_all_enrichment_results.xlsx`, `Rupture_vs_Erosion__robust_results.xlsx`, `Optico_revisions/plots/huCIRA/Heatmap_*.pdf`


In [ ]:
import os
OPTICO_DIR = os.environ.get("OPTICO_DIR", os.path.expanduser("~/OPTICO"))  # project folder


In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import hucira as hc

import warnings
from tqdm import TqdmWarning

warnings.simplefilter("ignore", TqdmWarning)

import time
start = time.time()

In [ ]:
#### Load the human cytokine dictionary

df_hcd_all = hc.load_human_cytokine_dict()

print(f"All celltypes in dictionary:\n  {df_hcd_all.celltype.unique()}")
print("\n")
print(f"All cytokines in dictionary:\n  {df_hcd_all.cytokine.unique()}")

df_hcd_all.head()

In [ ]:
#### Load the query adata object

adata = sc.read_h5ad(os.path.join(OPTICO_DIR, "Optico_revisions/data/optico_final.h5ad"))
adata

In [ ]:
adata.obs.columns

In [ ]:
# Enter celltype column name and condition column name
your_celltype_colname = "Res2"
your_contrast_colname = "disease"


# Check data size (Are the conditions you want to contrast of comparable size?)
adata.obs[your_contrast_colname].value_counts()

In [ ]:
# Enrichment analysis needs two main information from query adata: cell types and disease conditions.
# They have to be chosen manually, because annotation of objects differs.

print(f"All celltypes in query data:\n  {sorted(adata.obs[your_celltype_colname].unique())}\n")
print(f"All experimental states (contrasts/conditions) in query data:\n  {sorted(adata.obs[your_contrast_colname].unique())}\n")

In [ ]:
### Query data needs to be pre-processed

# store counts safely
adata.layers["counts"] = adata.X.copy()

# log-normalize in adata.X
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# optional: keep lognorm too
adata.layers["lognorm"] = adata.X.copy()

adata

In [ ]:
#### Create celltype_combos. An input parameter for the enrichment analysis.

# Your cell types (from Optico)
adata_celltypes = [
        "NK",
        "CD4_Naive",
        "CD4_TCM_TEM",
        "CD8_Naive",
        "CD8_TCM_TEM",
        "Treg",
        "MAIT",
        "CD14_Mono",
        "CD16_Mono",
        "B_Naive",
        "B_intermediate",
        "B_Memory",
        "B_class_switch",
        "Plasmablast",
        "HSPC", 
        "cDC", 
        "ASDC"
]

# Dictionary cell types (HuCIRA human cytokine dictionary)
hcd_celltypes = [
        "NK",
        "CD4_Naive_T_cell",
        "CD4_Memory_T_cell",
        "CD8_Naive_T_cell",
        "CD8_Memory_T_cell",
        "Treg",
        "MAIT",
        "CD14_Mono",
        "CD16_Mono",
        "Naive_B_cell",
        "Intermediate_B_cell",
        "B_cell",
        "B_cell",
        "Plasmablast",
        "HSPC", 
        "cDC", 
        "cDC"
]

celltype_combos = hc.create_celltype_combos(adata_celltypes, hcd_celltypes)
celltype_combos

In [ ]:
#### Define conditions of interest:

contrasts = [("Rupture", "Erosion")]

In [ ]:
#### Run enrichment analysis on a single celltype and condition:


enrichment_results = hc.run_one_enrichment_test(
    adata=adata,
    df=df_hcd_all,
    contrasts_combo=contrasts,
    celltype_combo=celltype_combos[0],
    contrast_column=your_contrast_colname,
    celltype_column=your_celltype_colname,
    direction="upregulated",
    threshold_lfc=0.8,
    threshold_expression=0.05,
    threshold_pval=0.01
)

# Look at all enrichment results (reduced view to columns of interest, not all statistics)
enrichment_results[
    [
        "celltype_combo",
        "cytokine",
        "contrast",
        "direction",
        "ES",
        "NES",
        "NOM p-val",
        "FDR q-val",
        "FWER p-val",
        "frac_shared_genes_signature",
    ]
].sort_values("NES", ascending=False)

In [ ]:
#### Lastly, get robust and significant results from previous enrichment_results

robust_results_dict = hc.get_robust_significant_results(
    results=enrichment_results,
    alphas=[0.1, 0.05, 0.01],
    threshold_valid=0.1,
    threshold_below_alpha=0.9,
    display_df_nicely=True,
)

In [ ]:
#### Run more robust enrichment analysis on a several celltypes and conditions:

all_enrichment_results = hc.run_all_enrichment_test(
    adata=adata,
    df=df_hcd_all,
    contrasts_combo=contrasts,
    celltype_combos=celltype_combos,
    contrast_column=your_contrast_colname,
    celltype_column=your_celltype_colname,
    direction="upregulated",
    threshold_lfc=[0.8, 1],
    threshold_expression=[0.05],
    min_size=1,        # try 1–5 depending on how sparse overlap is
    max_size=2000
)

# Look at all enrichment results (reduced view to columns of interest, not all statistics)
all_enrichment_results[
    [
        "celltype_combo",
        "cytokine",
        "contrast",
        "direction",
        "ES",
        "NES",
        "NOM p-val",
        "FDR q-val",
        "FWER p-val",
        "frac_shared_genes_signature",
    ]
].sort_values("NES", ascending=False)

In [ ]:
all_enrichment_results.to_excel(
    os.path.join(OPTICO_DIR, "Optico_revisions/tables/HuCIRA_all_enrichment_results.xlsx"),
    index=False
)

In [ ]:
#### Create heatmap to visualize up-/down regulation of cytokines in all queried cell types for one contrast comparison.

robust_results_dict = hc.get_robust_significant_results(
    results=all_enrichment_results,
    alphas=[0.1, 0.05, 0.01],
    threshold_valid=0.1,
    threshold_below_alpha=0.9,
    display_df_nicely=True,
)


contrast_one = all_enrichment_results.contrast.unique()[0]  # Can be looked up manually
print(f"Contrast that was plotted in this example: {contrast_one}")

hc.plot_significant_results(
    results_pivot=robust_results_dict[contrast_one][0],
    df_annot=robust_results_dict[contrast_one][1],
    fontsize=6,
    save_fig=True,
    fig_path=os.path.join(OPTICO_DIR, "Optico_revisions/plots/huCIRA"),
)

In [ ]:
import os

outdir = os.path.join(OPTICO_DIR, "Optico_revisions/tables/")
os.makedirs(outdir, exist_ok=True)

contrast_one = all_enrichment_results.contrast.unique()[0]

results_pivot = robust_results_dict[contrast_one][0]
df_annot      = robust_results_dict[contrast_one][1]

# One Excel with two sheets
xlsx_path = os.path.join(outdir, f"{contrast_one}__robust_results.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    results_pivot.to_excel(writer, sheet_name="heatmap_matrix")
    df_annot.to_excel(writer, sheet_name="annotations", index=False)

print("Wrote:", xlsx_path)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# -----------------------------
# Get contrast + pivot table
# -----------------------------
contrast_one = all_enrichment_results.contrast.unique()[0]
results_pivot = robust_results_dict[contrast_one][0]

if not isinstance(results_pivot, pd.DataFrame):
    results_pivot = pd.DataFrame(results_pivot)

# -----------------------------
# Transpose for landscape view
# -----------------------------
results_pivot = results_pivot.T

# -----------------------------
# Custom colormap: black → white → red
# -----------------------------
cmap = LinearSegmentedColormap.from_list(
    "black_white_red",
    ["#904011", "white", "#0F1E5E"],
    N=256
)

cmap.set_bad(color="#d9d9d9")

# Symmetric color scale
max_abs = np.nanmax(np.abs(results_pivot.values))

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(32, 8))

sns.heatmap(
    results_pivot,
    ax=ax,
    cmap=cmap,
    center=0,
    vmin=-max_abs,
    vmax=max_abs,
    linewidths=4.5,
    linecolor="white",
    cbar_kws={"label": "Enrichment (signed)"},
)

ax.set_title(f"Significant cytokines — {contrast_one}", fontsize=8)
ax.set_xlabel("Cytokine")
ax.set_ylabel("Cell type")

ax.tick_params(axis="x", labelrotation=90, labelsize=7)
ax.tick_params(axis="y", labelrotation=0, labelsize=7)

plt.tight_layout()

# -----------------------------
# Save
# -----------------------------
fig.savefig(
    os.path.join(OPTICO_DIR, "Optico_revisions/plots/huCIRA/Heatmap_robust_landscape.pdf"),
    dpi=300,
    bbox_inches="tight",
)

plt.close(fig)


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

contrast_one = all_enrichment_results.contrast.unique()[0]
results_pivot = robust_results_dict[contrast_one][0]
if not isinstance(results_pivot, pd.DataFrame):
    results_pivot = pd.DataFrame(results_pivot)

results_pivot = results_pivot.T

cmap = LinearSegmentedColormap.from_list(
    "black_white_red",
    ["#904011", "white", "#0F1E5E"],
    N=256
)

cmap.set_bad(color="#d9d9d9")

max_abs = np.nanmax(np.abs(results_pivot.values))

# fill NaNs for clustering
X = results_pivot.fillna(0)

# row order
row_link = linkage(pdist(X.values, metric="euclidean"), method="average")
row_order = leaves_list(row_link)

# col order
col_link = linkage(pdist(X.values.T, metric="euclidean"), method="average")
col_order = leaves_list(col_link)

# reorder
X_ord = results_pivot.iloc[row_order, col_order]

fig, ax = plt.subplots(figsize=(18, 6))
sns.heatmap(
    X_ord,
    ax=ax,
    cmap=cmap,
    center=0,
    vmin=-max_abs,
    vmax=max_abs,
    linewidths=2,
    linecolor="white",
    cbar_kws={"label": "Enrichment (signed)"},
)

ax.set_title(f"Significant cytokines — {contrast_one} (clustered order)", fontsize=8)
ax.set_xlabel("Cytokine")
ax.set_ylabel("Cell type")
ax.tick_params(axis="x", labelrotation=90, labelsize=7)
ax.tick_params(axis="y", labelrotation=0, labelsize=7)

plt.tight_layout()
fig.savefig(
    os.path.join(OPTICO_DIR, "Optico_revisions/plots/huCIRA/Heatmap_robust_landscape_clustered_order_only.pdf"),
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)